In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD


CSV_PATH = "data/city_lifestyle_dataset.csv"  
OUT_FILE = "outputs/truncated_svd_emb_2d"

In [10]:
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (300, 10)
Columns: ['city_name', 'country', 'population_density', 'avg_income', 'internet_penetration', 'avg_rent', 'air_quality_index', 'public_transport_score', 'happiness_score', 'green_space_ratio']


,city_name,country,population_density,avg_income,internet_penetration,avg_rent,air_quality_index,public_transport_score,happiness_score,green_space_ratio
0,Old Vista,Europe,2775,3850,86.4,1310,43,52.0,8.5,23.8
1,Beachport,Europe,3861,3700,78.1,1330,42,62.8,8.1,33.1
2,Valleyborough,Europe,2562,4310,80.1,1330,39,73.2,8.5,40.2
3,City,Europe,3192,3970,81.2,1480,60,49.2,8.5,43.6
4,Falls,Europe,3496,4320,100.0,1510,64,93.7,8.5,42.5


In [ ]:
# On exclut les identifiants / catégorielles
drop_cols = [c for c in ["city_name", "country", "region"] if c in df.columns]
X = df.drop(columns=drop_cols)

print("Numeric features used:", X.columns.tolist())
display(X.describe().T)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_scaled)

# Variance expliquée 
explained = svd.explained_variance_ratio_
print("Explained variance ratio:", explained)
print("Total explained variance (2D):", float(np.sum(explained)))

In [ ]:
color_col = "country"


plt.figure(figsize=(10, 6))

cats = df[color_col].astype(str)
# mapping catégories -> ints pour colormap
cat_codes, uniques = pd.factorize(cats)
sc = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=cat_codes, alpha=0.7, s=50)
cbar = plt.colorbar(sc)
cbar.set_ticks(range(len(uniques)))
cbar.set_ticklabels(list(uniques))
cbar.set_label(color_col)


plt.xlabel(f"SVD-1 ({explained[0]:.2%})")
plt.ylabel(f"SVD-2 ({explained[1]:.2%})")
plt.title("Projection 2D — TruncatedSVD")
plt.tight_layout()
plt.show()


In [ ]:
df_2d = pd.DataFrame({
    "svd1": X_2d[:, 0],
    "svd2": X_2d[:, 1],
})
df_2d

In [ ]:
df_2d.to_csv(OUT_FILE, index=False, encoding="utf-8")
print("Export saved to:", OUT_FILE)